In [1]:
# Import libraries for data preprocessing
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# Import libraries for ML
import torch
import torch.nn as nn
from torch.optim import SGD
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

2. For this problem, you will explore different methods for encoding categorical random variables. We have
provided a dataset retail.csv, which contains data on retail transactions. We’ve cleaned up the data a
bit so you only have two features: StockCode and CustomerID, both of which are categorical. (You may
need to convert them to categorical data types in your code for the encoding methods to work properly.)
The target is Quantity, which is a positive integer. Note that Quantity is very heavy-tailed (most values
are small, but a few are enormous), so think about what loss function and/or target transformation is
appropriate.


(a) Start by exploring the cardinality of the two features. How many unique values are there for each
feature? What does this suggest about how you might encode them? Is one-hot encoding feasible?
To prepare for the next steps, split the data into training, validation, and test sets. The validation
set will allow you to use early stopping during training for the neural network models.

In [2]:
# Load dataset
df = pd.read_csv("retail.csv")
df.head()

,Quantity,StockCode,CustomerID
0,6,85123A,17850.0
1,6,71053,17850.0
2,8,84406B,17850.0
3,6,84029G,17850.0
4,6,84029E,17850.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 397924 entries, 0 to 397923
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Quantity    397924 non-null  int64  
 1   StockCode   397924 non-null  str    
 2   CustomerID  397924 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 11.0 MB


In [4]:
df.nunique()

Quantity       302
StockCode     3665
CustomerID    4339
dtype: int64

In [5]:
# Split to X and Y sets
X = df.drop(columns=["Quantity"])
y = df["Quantity"]

print(X.shape)
print(y.shape)

(397924, 2)
(397924,)


In [6]:
# Split into training and test sets
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.15 / 0.85, random_state=42)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(278546, 2)
(59689, 2)
(59689, 2)
(278546,)
(59689,)
(59689,)


(b) Try target encoding. For each unique value of the feature, replace it with the mean of the target
for that value (from the training set). You can use TargetEncoder from the category
encoders package. Using an appropriate loss function, fit a model of your choice to the data using the
target-encoded features. Evaluate the model on the test set.

In [7]:
# Transform y set using log1p b/c of right skew
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
y_test_log = np.log1p(y_test)

# Fit target encoder to training sets
enc = TargetEncoder(cols=['StockCode', 'CustomerID']).fit(X_train, y_train_log)

# Transform all X sets
X_train_enc = enc.transform(X_train)
X_val_enc = enc.transform(X_val)
X_test_enc = enc.transform(X_test)

X_train_enc.info()
print()
X_val_enc.info()
print()
X_test_enc.info()

<class 'pandas.DataFrame'>
Index: 278546 entries, 256028 to 234181
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   StockCode   278546 non-null  float64
 1   CustomerID  278546 non-null  float64
dtypes: float64(2)
memory usage: 6.4 MB

<class 'pandas.DataFrame'>
Index: 59689 entries, 46385 to 121564
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   StockCode   59689 non-null  float64
 1   CustomerID  59689 non-null  float64
dtypes: float64(2)
memory usage: 1.4 MB

<class 'pandas.DataFrame'>
Index: 59689 entries, 353660 to 91327
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   StockCode   59689 non-null  float64
 1   CustomerID  59689 non-null  float64
dtypes: float64(2)
memory usage: 1.4 MB


In [8]:
# Train linear regression model, which uses MSE for loss function
lr = LinearRegression()
lr.fit(X_train_enc,y_train_log)

# Make predictions
y_pred_log = lr.predict(X_test_enc)

# Evaluate MSE on log scale
mse_log = mean_squared_error(y_test_log, y_pred_log)

# Convert predictions back to original scale
y_pred = np.expm1(y_pred_log)

# Evaluate on original scale
mae = mean_absolute_error(y_test, y_pred)

print("Test MSE (log scale):", mse_log)
print("Test MAE (original scale):", mae)

Test MSE (log scale): 0.3624484778226378
Test MAE (original scale): 7.125435507011995


c) Try a neural network model with an embedding layer for each of the two features. Choose a
reasonable embedding dimension. Fit this model to the data. Evaluate the model on the test set.
How does it compare to the target encoding model?

In [9]:
# Map categorical strings/codes to integer indices for PyTorch embeddings
stock_mapping = {val: idx for idx, val in enumerate(X_train['StockCode'].unique())}
customer_mapping = {val: idx for idx, val in enumerate(X_train['CustomerID'].unique())}

# Convert features to tensors
X_train_tensor_stock = torch.tensor(X_train['StockCode'].map(stock_mapping).to_numpy(), dtype=torch.long)
X_train_tensor_customer = torch.tensor(X_train['CustomerID'].map(customer_mapping).to_numpy(), dtype=torch.long)
X_val_tensor_stock = torch.tensor(X_val['StockCode'].map(stock_mapping).fillna(0).to_numpy(), dtype=torch.long)
X_val_tensor_customer = torch.tensor(X_val['CustomerID'].map(customer_mapping).fillna(0).to_numpy(), dtype=torch.long)
X_test_tensor_stock = torch.tensor(X_test['StockCode'].map(stock_mapping).fillna(0).values, dtype=torch.long)
X_test_tensor_customer = torch.tensor(X_test['CustomerID'].map(customer_mapping).fillna(0).to_numpy(), dtype=torch.long)

# Convert targets to tensors
y_train_tensor = torch.tensor(y_train_log.to_numpy(), dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_log.to_numpy(), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_log.to_numpy(), dtype=torch.float32)

# Create datasets
train_dataset = TensorDataset(X_train_tensor_stock, X_train_tensor_customer, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor_stock, X_val_tensor_customer, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor_stock, X_test_tensor_customer, y_test_tensor)

# Create dataloaders
batch_sz = 128
train_loader = DataLoader(train_dataset, batch_size=batch_sz, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_sz, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_sz, shuffle=False)

In [10]:
class embeddingNN(nn.Module):
    def __init__(self, n_stock, n_customer, hidden_dim = 64, embedding_dim = 8):
        super().__init__()

        # Set up embedding layers
        self.stock_embedding = nn.Embedding(n_stock, embedding_dim)
        self.customer_embedding = nn.Embedding(n_customer, embedding_dim)

        # Set up architecture
        self.net = nn.Sequential(
            nn.Linear(embedding_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, stock, customer):
        # Map to embedding vectors
        stock_emb = self.stock_embedding(stock)
        customer_emb = self.customer_embedding(customer)

        # Concatenate to 1 vector
        x = torch.concat([stock_emb, customer_emb], dim=1)
        return self.net(x).squeeze(1)

# Set up model and optimizer
model = embeddingNN(len(stock_mapping) + 1, len(customer_mapping) + 1).to(device='xpu')
optimizer = SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [11]:
epochs = 100
best_val_loss = float('inf')
patience = 5
patience_counter = 0

for epoch in range(epochs):
    # Training MLP
    model.train()
    train_losses = []

    for X_stock_batch, X_customer_batch, y_batch in train_loader:
        # Move batch to XPU
        X_stock_batch = X_stock_batch.to(device='xpu')
        X_customer_batch = X_customer_batch.to(device='xpu')
        y_batch = y_batch.to(device='xpu')

        optimizer.zero_grad()

        # Calculate loss
        pred_train = model(X_stock_batch, X_customer_batch)
        loss_train = criterion(pred_train, y_batch)

        # Update weights
        loss_train.backward()
        optimizer.step()

        train_losses.append(loss_train.item())

    avg_train_loss = np.mean(train_losses)

    # Validation
    model.eval()
    val_losses = []

    with torch.no_grad():
        for X_stock_batch, X_customer_batch, y_batch in val_loader:
            # Move batch to XPU
            X_stock_batch = X_stock_batch.to(device='xpu')
            X_customer_batch = X_customer_batch.to(device='xpu')
            y_batch = y_batch.to(device='xpu')
                    
            # Calculate validation loss
            pred_val = model(X_stock_batch, X_customer_batch)
            loss_val = criterion(pred_val, y_batch)

            val_losses.append(loss_val.item())

    avg_val_loss = np.mean(val_losses)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d} | Train MSE: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_embedding_nn.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

model.load_state_dict(torch.load('best_embedding_nn.pth'))

Epoch 10 | Train MSE: 0.8112 | Val MSE: 0.8114
Epoch 20 | Train MSE: 0.6765 | Val MSE: 0.7044
Epoch 30 | Train MSE: 0.6048 | Val MSE: 0.6372
Early stopping triggered.


<All keys matched successfully>

In [12]:
# Evaluate on test set
model.eval()
pred_test_log = []

with torch.no_grad():
    for X_stock_batch, X_customer_batch, y_batch in test_loader:
        # Move batch to XPU
        X_stock_batch = X_stock_batch.to(device='xpu')
        X_customer_batch = X_customer_batch.to(device='xpu')

        # Make predictions
        pred_test_log.extend(model(X_stock_batch, X_customer_batch).cpu().numpy())

# Evaluate MSE on log scale
mse_log_nn = mean_squared_error(y_test_log, pred_test_log)

# Convert back to original scale
pred_test = np.expm1(pred_test_log)

# Evaluate MAE on original scale
mae_nn = mean_absolute_error(y_test, pred_test)

print("Target Encoder with Linear Regression")
print("Test MSE (log scale):", mse_log) # want to minimize
print("Test MAE (original scale):", mae) # want to minimize

print("\nNN with Embedding Layer")
print("Test MSE (log scale):", mse_log_nn) # want to minimize
print("Test MAE (original scale):", mae_nn) # want to minimize

Target Encoder with Linear Regression
Test MSE (log scale): 0.3624484778226378
Test MAE (original scale): 7.125435507011995

NN with Embedding Layer
Test MSE (log scale): 0.6102903816301641
Test MAE (original scale): 8.658620834350586


d) What if we combine the two? That is, use both the target encoded features and the embeddings as
input to the neural network. Fit this model to the data. Evaluate the model on the test set. How
does it compare to the other two models?

In [13]:
# Convert target encoded features to tensors
X_train_enc_tensor = torch.tensor(X_train_enc.to_numpy(), dtype=torch.float32)
X_val_enc_tensor = torch.tensor(X_val_enc.to_numpy(), dtype=torch.float32)
X_test_enc_tensor = torch.tensor(X_test_enc.to_numpy(), dtype=torch.float32)

# Create datasets
train_dataset = TensorDataset(X_train_tensor_stock, X_train_tensor_customer, X_train_enc_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor_stock, X_val_tensor_customer, X_val_enc_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor_stock, X_test_tensor_customer, X_test_enc_tensor, y_test_tensor)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_sz, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_sz, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_sz, shuffle=False)

In [14]:
class combinedNN(nn.Module):
    def __init__(self, n_stock, n_customer, hidden_dim = 64, embedding_dim = 8):
        super().__init__()

        # Set up embedding layers
        self.stock_embedding = nn.Embedding(n_stock, embedding_dim)
        self.customer_embedding = nn.Embedding(n_customer, embedding_dim)

        # Set up architecture
        self.net = nn.Sequential(
            nn.Linear(embedding_dim * 2 + 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, stock, customer, target_encoded):
        # Map to embedding vectors
        stock_emb = self.stock_embedding(stock)
        customer_emb = self.customer_embedding(customer)
        
        # Concatenate to 1 vector
        x = torch.concat([stock_emb, customer_emb, target_encoded], dim=1)
        return self.net(x).squeeze(1)

# Set up model and optimizer
model_combined = combinedNN(len(stock_mapping) + 1, len(customer_mapping) + 1).to(device='xpu')
optimizer = SGD(model_combined.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [15]:
epochs = 100
best_val_loss = float('inf')
patience = 5
patience_counter = 0

for epoch in range(epochs):
    # Training MLP
    model_combined.train()
    train_losses = []

    for X_stock_batch, X_customer_batch, X_enc_batch, y_batch in train_loader:
        # Move batch to XPU
        X_stock_batch = X_stock_batch.to(device='xpu')
        X_customer_batch = X_customer_batch.to(device='xpu')
        X_enc_batch = X_enc_batch.to(device='xpu')
        y_batch = y_batch.to(device='xpu')

        optimizer.zero_grad()

        # Calculate loss
        pred_train = model_combined(X_stock_batch, X_customer_batch, X_enc_batch)
        loss_train = criterion(pred_train, y_batch)

        # Update weights
        loss_train.backward()
        optimizer.step()

        train_losses.append(loss_train.item())

    avg_train_loss = np.mean(train_losses)

    # Validation
    model_combined.eval()
    val_losses = []

    with torch.no_grad():
        for X_stock_batch, X_customer_batch, X_enc_batch, y_batch in val_loader:
            # Move batch to XPU
            X_stock_batch = X_stock_batch.to(device='xpu')
            X_customer_batch = X_customer_batch.to(device='xpu')
            X_enc_batch = X_enc_batch.to(device='xpu')
            y_batch = y_batch.to(device='xpu')
                    
            # Calculate validation loss
            pred_val = model_combined(X_stock_batch, X_customer_batch, X_enc_batch)
            loss_val = criterion(pred_val, y_batch)

            val_losses.append(loss_val.item())

    avg_val_loss = np.mean(val_losses)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d} | Train MSE: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model_combined.state_dict(), 'best_combined_nn.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

model_combined.load_state_dict(torch.load('best_combined_nn.pth'))

Epoch 10 | Train MSE: 0.3459 | Val MSE: 0.3581
Epoch 20 | Train MSE: 0.3398 | Val MSE: 0.3522
Early stopping triggered.


<All keys matched successfully>

In [16]:
# Evaluate on test set
model_combined.eval()
pred_test_combined_log = []

with torch.no_grad():
    for X_stock_batch, X_customer_batch, X_enc_batch, y_batch in test_loader:
        # Move batch to XPU
        X_stock_batch = X_stock_batch.to(device='xpu')
        X_customer_batch = X_customer_batch.to(device='xpu')
        X_enc_batch = X_enc_batch.to(device='xpu')

        # Make predictions
        pred_test_combined_log.extend(model_combined(X_stock_batch, X_customer_batch, X_enc_batch).cpu().numpy())

# Evaluate MSE on log scale
mse_log_combined_nn = mean_squared_error(y_test_log, pred_test_combined_log)

# Convert back to original scale
pred_combined_test = np.expm1(pred_test_combined_log)

# Evaluate MAE on original scale
mae_combined_nn = mean_absolute_error(y_test, pred_combined_test)

print("Target Encoder with Linear Regression")
print("Test MSE (log scale):", mse_log) # want to minimize
print("Test MAE (original scale):", mae) # want to minimize

print("\nNN with Embedding Layer")
print("Test MSE (log scale):", mse_log_nn) # want to minimize
print("Test MAE (original scale):", mae_nn) # want to minimize

print("\nCombined NN")
print("Test MSE (log scale):", mse_log_combined_nn) # want to minimize
print("Test MAE (original scale):", mae_combined_nn) # want to minimize

Target Encoder with Linear Regression
Test MSE (log scale): 0.3624484778226378
Test MAE (original scale): 7.125435507011995

NN with Embedding Layer
Test MSE (log scale): 0.6102903816301641
Test MAE (original scale): 8.658620834350586

Combined NN
Test MSE (log scale): 0.3499791097549076
Test MAE (original scale): 7.0556745529174805
